# 06 — Baseline and BM25 Threshold Abstention

This notebook implements the first two decision strategies in the proposal. The same BM25 retrieval outputs and QA predictions are reused; only the abstention decision changes.

## 1. Load retrieval and QA predictions

This notebook implements two decision strategies from the proposal: (1) always attempt an answer and (2) abstain when the top BM25 score is below a threshold. Thresholds are selected on validation data only.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support

ROOT=Path.cwd()
if ROOT.name=="notebooks": ROOT=ROOT.parent
SPLITS=ROOT/"data"/"splits"
RESULTS=ROOT/"results"

def load_experiment_data(name):
    q=pd.read_csv(SPLITS/f"{name}.csv")
    q["is_answerable"]=q["is_answerable"].astype(bool)
    ret=pd.read_csv(RESULTS/f"{name}_retrieval_features.csv")
    qa=pd.read_csv(RESULTS/f"{name}_qa_predictions.csv")
    return q.merge(ret,on="question_id",validate="one_to_one").merge(
        qa,on="question_id",validate="one_to_one"
    )

train=load_experiment_data("train")
val=load_experiment_data("validation")
test=load_experiment_data("test")
print(len(train),len(val),len(test))

## 2. Define the target and decision semantics

For this experiment, `is_answerable` is the SQuAD label for the associated source passage. The classifier/threshold predicts whether the example should be treated as answerable. This is a benchmark proxy for evidence sufficiency; it is not a new human annotation of the full retrieved top-k set.

A strategy returns either:
- `answer`: allow the fixed QA prediction to be shown;
- `abstain`: refuse because evidence is judged insufficient.

The baseline always allows the QA output, including an empty QA output if the model returns one.

In [ ]:
def always_answer_decision(df):
    return np.ones(len(df), dtype=bool)

def threshold_decision(df, threshold):
    return df["top_bm25_score"].to_numpy() >= threshold

print("Train BM25 score summary:")
display(train.top_bm25_score.describe())

## 3. Tune the BM25 threshold on validation

BM25 scores are not probabilities and their scale depends on the corpus and query. We search a set of candidate thresholds derived from validation-score quantiles. The selection criterion is answerability F1; you may choose a different criterion if your report explains the trade-off.

The test set is not used to choose the threshold.

In [ ]:
def binary_metrics(y_true, y_pred):
    p,r,f1,_=precision_recall_fscore_support(
        y_true,y_pred,average="binary",zero_division=0
    )
    return {"precision":p,"recall":r,"f1":f1}

y_val=val.is_answerable.astype(int).to_numpy()
quantiles=np.linspace(0,1,101)
candidates=np.unique(np.quantile(val.top_bm25_score,quantiles))

rows=[]
for threshold in candidates:
    pred=threshold_decision(val,threshold).astype(int)
    rows.append({"threshold":float(threshold),**binary_metrics(y_val,pred)})
threshold_results=pd.DataFrame(rows).sort_values("f1",ascending=False)
display(threshold_results.head(10))
BEST_THRESHOLD=float(threshold_results.iloc[0].threshold)
print("Selected validation threshold:",BEST_THRESHOLD)

## 4. Compare the two strategies on test

Only after the threshold is fixed do we evaluate the test split. We report answerability precision, recall and F1. Additional task-level measures (false-answer rate, false-abstention rate and answer quality) are computed in Notebook 08.

In [ ]:
y_test=test.is_answerable.astype(int).to_numpy()
baseline_pred=always_answer_decision(test).astype(int)
threshold_pred=threshold_decision(test,BEST_THRESHOLD).astype(int)

comparison=pd.DataFrame([
    {"strategy":"Always attempt (baseline)",**binary_metrics(y_test,baseline_pred)},
    {"strategy":"BM25 threshold",**binary_metrics(y_test,threshold_pred)},
])
display(comparison)
comparison.to_csv(RESULTS/"baseline_threshold_answerability_metrics.csv",index=False)

## 5. Inspect the threshold trade-off

As the threshold increases, coverage usually decreases. Plot validation precision and recall across candidate thresholds. Do not select a threshold using the test curve.

In [ ]:
plt.figure(figsize=(9,5))
plt.plot(threshold_results.threshold,threshold_results.precision,label="Precision")
plt.plot(threshold_results.threshold,threshold_results.recall,label="Recall")
plt.plot(threshold_results.threshold,threshold_results.f1,label="F1")
plt.axvline(BEST_THRESHOLD,linestyle="--",label="Selected threshold")
plt.xlabel("BM25 threshold")
plt.ylabel("Metric")
plt.title("Validation metrics across BM25 thresholds")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Save decisions for downstream evaluation

In [ ]:
for name, frame in [("validation",val),("test",test)]:
    out=frame[["question_id"]].copy()
    out["always_answer"]=True
    out["bm25_threshold_answer"]=frame.top_bm25_score >= BEST_THRESHOLD
    out.to_csv(RESULTS/f"{name}_threshold_decisions.csv",index=False)

print("Selected threshold:",BEST_THRESHOLD)
print("Saved validation and test decisions.")

## 7. Interpretation questions

1. Why is a threshold chosen on validation rather than test?
2. Why might one threshold not transfer to a different corpus?
3. What happens to false abstentions when the threshold is increased?
4. Why is a top BM25 score only a relevance signal, not proof that the passage supports the answer?

**Next:** Notebook 07 trains a lightweight answerability classifier.